# ASTE Model v2 — Improved (Aspect, Opinion, VA) Triplet Extraction

**Improvements over v1:**
1. **T5-base** (220M params) instead of T5-small (60M) — significantly better representation capacity
2. **Text preprocessing** — normalize whitespace, clean noisy tokens before feeding to model
3. **Cosine LR scheduler** with warmup ratio for smoother convergence
4. **Gradient accumulation** — effective batch size of 32 (4 steps × 8) for more stable gradients
5. **More epochs (15)** with patient early stopping via `load_best_model_at_end`
6. **Better beam search** — 5 beams + length penalty for higher quality generation
7. **Category is ignored** as instructed

## 0. Install Dependencies

In [ ]:
#!pip install transformers datasets sentencepiece protobuf accelerate -q

In [4]:
import json
import re
import os
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch: 2.10.0+cu128
GPU: Tesla T4


## 1. Configuration

In [5]:
# ── Paths (Kaggle) ──
DATA_DIR       = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE      = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE       = DATA_DIR / "laptop_train.jsonl"
MODEL_DIR      = Path("/kaggle/working/aste_t5base_model")
OUTPUT_FILE    = Path("/kaggle/working/predictions.jsonl")

# ── Improved Hyper-parameters ──
MODEL_NAME       = "t5-base"                     # 220M params (was t5-small 60M)
MAX_INPUT_LEN    = 256
MAX_TARGET_LEN   = 256
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 4                             # effective batch = 8 × 4 = 32
LEARNING_RATE    = 2e-4                           # slightly lower for t5-base
NUM_EPOCHS       = 15                             # more epochs with early stopping
WARMUP_RATIO     = 0.1                            # 10% warmup
VAL_SPLIT        = 0.1
LOGGING_STEPS    = 50
NUM_BEAMS        = 5                              # more beams for better generation
LENGTH_PENALTY   = 1.0                            # encourage full-length outputs

## 2. Text Preprocessing & Data Loading

In [6]:
def preprocess_text(text):
    """
    Clean and normalize text before feeding to the model.
    Inspired by ABSA best practices:
    - Normalize whitespace
    - Remove extra spaces
    - Keep case-sensitivity (required by assignment)
    """
    # Normalize various whitespace characters
    text = re.sub(r'[\t\n\r\f\v]+', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing
    text = text.strip()
    return text


def load_jsonl(filepath):
    """Load JSONL file -> list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


rest_data = load_jsonl(REST_FILE)
lap_data  = load_jsonl(LAP_FILE)
all_data  = rest_data + lap_data

print(f"Restaurant (REST): {len(rest_data)}")
print(f"Laptop     (LAP):  {len(lap_data)}")
print(f"Total:             {len(all_data)}")

Restaurant (REST): 2284
Laptop     (LAP):  4076
Total:             6360


### 2.1 Linearize Triplets for Seq2Seq

- **Input**: `"extract triplets: <preprocessed review text>"`
- **Target**: `"( Aspect | Opinion | V#A ) ; ( Aspect | Opinion | V#A ) ; ..."`

Category is **ignored**.

In [7]:
def record_to_seq2seq(record):
    """
    Convert a single JSONL record to (input_text, target_text) for T5.
    Applies text preprocessing. Category is ignored.
    """
    text = preprocess_text(record["Text"])
    input_text = f"extract triplets: {text}"

    triplet_strs = []
    for quad in record["Quadruplet"]:
        aspect  = quad.get("Aspect", "NULL")
        opinion = quad.get("Opinion", "NULL")
        va      = quad.get("VA", "5.00#5.00")
        triplet_strs.append(f"( {aspect} | {opinion} | {va} )")

    target_text = " ; ".join(triplet_strs)
    return input_text, target_text


# Build pairs
pairs = []
for rec in all_data:
    inp, tgt = record_to_seq2seq(rec)
    pairs.append({"id": rec["ID"], "input": inp, "target": tgt})

print(f"Total seq2seq pairs: {len(pairs)}")
print()
print("Example:")
print(f"  INPUT:  {pairs[0]['input'][:120]}...")
print(f"  TARGET: {pairs[0]['target'][:120]}...")

Total seq2seq pairs: 6360

Example:
  INPUT:  extract triplets: ca n ' t wait wait for my next visit ....
  TARGET: ( NULL | NULL | 6.75#6.38 )...


### 2.2 Train / Validation Split

In [8]:
random.shuffle(pairs)
split_idx = int(len(pairs) * (1 - VAL_SPLIT))
train_pairs = pairs[:split_idx]
val_pairs   = pairs[split_idx:]

print(f"Train: {len(train_pairs)}")
print(f"Val:   {len(val_pairs)}")

Train: 5724
Val:   636


## 3. Tokenization & Dataset

In [9]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=True)

class ASTEDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input_len, max_target_len):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_enc = self.tokenizer(
            pair["input"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        target_enc = self.tokenizer(
            pair["target"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = target_enc["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels":         labels,
        }

train_dataset = ASTEDataset(train_pairs, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_dataset   = ASTEDataset(val_pairs,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

print(f"Train samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")

sample = train_dataset[0]
print(f"input_ids shape:  {sample['input_ids'].shape}")
print(f"labels shape:     {sample['labels'].shape}")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train samples: 5724
Val   samples: 636
input_ids shape:  torch.Size([256])
labels shape:     torch.Size([256])


## 4. Model & Training (T5-base)

In [10]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: t5-base
Parameters: 222,903,552


In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,  # effective batch = 32
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,                     # 10% warmup
    lr_scheduler_type="cosine",                     # cosine annealing
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=3,
    eval_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print("Trainer ready.")
print(f"  Model:           {MODEL_NAME} ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Device:          {DEVICE}")
print(f"  FP16:            {training_args.fp16}")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  LR:              {LEARNING_RATE} (cosine annealing)")
print(f"  Warmup:          {warmup_steps} steps ({WARMUP_RATIO*100:.0f}%)")
print(f"  Total steps:     {total_steps}")
print(f"  Train samples:   {len(train_dataset)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready.
  Model:           t5-base (222,903,552 params)
  Device:          cuda
  FP16:            True
  Epochs:          15
  Batch size:      8 (effective: 32)
  LR:              0.0002 (cosine annealing)
  Warmup:          267 steps (10%)
  Total steps:     2670
  Train samples:   5724


In [ ]:
# ══════════════════════════════════════════════════════
#  TRAINING  (Tesla T4 GPU: ~30-60 min)
# ══════════════════════════════════════════════════════
trainer.train()
print("\n✅ Training complete!")

In [ ]:
# Save the best model
trainer.save_model(str(MODEL_DIR / "best"))
tokenizer.save_pretrained(str(MODEL_DIR / "best"))
print(f"Model saved to: {MODEL_DIR / 'best'}")

## 5. Inference — Parse Generated Triplets

In [ ]:
def parse_triplets(generated_text):
    """
    Parse linearized output back into structured triplets.
    Expected format: ( Aspect | Opinion | V#A ) ; ...
    """
    triplets = []
    pattern = r"\(\s*(.+?)\s*\|\s*(.+?)\s*\|\s*([\d.]+#[\d.]+)\s*\)"
    for match in re.finditer(pattern, generated_text):
        aspect  = match.group(1).strip()
        opinion = match.group(2).strip()
        va      = match.group(3).strip()

        # Clamp VA to valid range [1.00, 9.00]
        try:
            v, a = va.split("#")
            v, a = float(v), float(a)
            v = max(1.0, min(9.0, v))
            a = max(1.0, min(9.0, a))
            va = f"{v:.2f}#{a:.2f}"
        except:
            va = "5.00#5.00"

        triplets.append({
            "Aspect":  aspect,
            "Opinion": opinion,
            "VA":      va,
        })
    return triplets

# Quick test
test_str = "( Indian food | average to good | 6.75#6.38 ) ; ( delivery | terrible | 2.88#6.62 )"
print("Parse test:", parse_triplets(test_str))

In [ ]:
def predict_triplets(texts, model, tokenizer, max_len=256, batch_size=16,
                     num_beams=5, length_penalty=1.0):
    """
    Generate triplet predictions with improved beam search.
    """
    model.eval()
    all_triplets = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs = [f"extract triplets: {preprocess_text(t)}" for t in batch_texts]

        encoded = tokenizer(
            inputs,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_length=max_len,
                num_beams=num_beams,
                length_penalty=length_penalty,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for gen_text in decoded:
            triplets = parse_triplets(gen_text)
            all_triplets.append(triplets)

    return all_triplets

print("Inference function ready.")

## 6. Evaluate on Validation Set

In [ ]:
val_texts  = [p["input"].replace("extract triplets: ", "") for p in val_pairs]
val_golds  = [p["target"] for p in val_pairs]
val_ids    = [p["id"] for p in val_pairs]

val_preds = predict_triplets(val_texts, model, tokenizer,
                             max_len=MAX_TARGET_LEN,
                             num_beams=NUM_BEAMS,
                             length_penalty=LENGTH_PENALTY)

# Show examples
for i in range(min(8, len(val_preds))):
    print(f"\n{'='*70}")
    print(f"ID:    {val_ids[i]}")
    print(f"Text:  {val_texts[i][:100]}...")
    print(f"Gold:  {val_golds[i][:100]}...")
    print(f"Pred:  {val_preds[i]}")

In [ ]:
def compute_metrics(val_pairs, val_preds):
    """Compute span extraction F1 and VA MAE."""
    total_gold = 0
    total_pred = 0
    correct    = 0
    va_errors  = []

    for pair, preds in zip(val_pairs, val_preds):
        gold_triplets = parse_triplets(pair["target"])
        total_gold += len(gold_triplets)
        total_pred += len(preds)

        gold_set = {(g["Aspect"].lower(), g["Opinion"].lower()): g["VA"] for g in gold_triplets}
        for p in preds:
            key = (p["Aspect"].lower(), p["Opinion"].lower())
            if key in gold_set:
                correct += 1
                try:
                    gv, ga = gold_set[key].split("#")
                    pv, pa = p["VA"].split("#")
                    va_errors.append(abs(float(gv) - float(pv)) + abs(float(ga) - float(pa)))
                except:
                    pass

    prec = correct / total_pred if total_pred else 0
    rec  = correct / total_gold if total_gold else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    avg_va = np.mean(va_errors) if va_errors else float('inf')

    print(f"\n{'='*50}")
    print(f"Span Extraction:")
    print(f"  Gold: {total_gold}  Pred: {total_pred}  Correct: {correct}")
    print(f"  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    print(f"VA Error (matched): {avg_va:.4f}  (n={len(va_errors)})")

compute_metrics(val_pairs, val_preds)

## 7. Generate Predictions on Test Data

Place the test JSONL in your Kaggle dataset and update the path below.

In [ ]:
# ══════════════════════════════════════════════════════
#  TEST INFERENCE
# ══════════════════════════════════════════════════════

TEST_FILE = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp/test.jsonl")

if TEST_FILE.exists():
    # Load the best model (in case kernel was restarted)
    best_path = MODEL_DIR / "best"
    if best_path.exists():
        model = T5ForConditionalGeneration.from_pretrained(str(best_path)).to(DEVICE)
        tokenizer = T5Tokenizer.from_pretrained(str(best_path), legacy=True)
        print(f"Loaded model from: {best_path}")
    else:
        print("Using model already in memory.")

    # Load test data
    test_data = load_jsonl(TEST_FILE)
    print(f"Test records: {len(test_data)}")

    test_ids   = [r["ID"] for r in test_data]
    test_texts = [r["Text"] for r in test_data]

    # Predict with improved beam search
    test_preds = predict_triplets(
        test_texts, model, tokenizer,
        max_len=MAX_TARGET_LEN,
        num_beams=NUM_BEAMS,
        length_penalty=LENGTH_PENALTY,
    )

    # Write output JSONL
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for rid, triplets in zip(test_ids, test_preds):
            out = {"ID": rid, "Triplet": triplets}
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"\n✅ Predictions written to: {OUTPUT_FILE}")
    print(f"   Total records: {len(test_ids)}")

    # Preview first 3
    with open(OUTPUT_FILE, "r") as f:
        for i, line in enumerate(f):
            if i >= 3: break
            print(json.loads(line))
else:
    print(f"⚠️  Test file not found: {TEST_FILE}")
    print("   Upload test JSONL to your Kaggle dataset and re-run this cell.")

## 8. Self-Check on Training Samples

In [ ]:
train_sample_texts = [p["input"].replace("extract triplets: ", "") for p in train_pairs[:20]]
train_sample_ids   = [p["id"] for p in train_pairs[:20]]
train_sample_golds = [p["target"] for p in train_pairs[:20]]

train_sample_preds = predict_triplets(
    train_sample_texts, model, tokenizer,
    max_len=MAX_TARGET_LEN,
    num_beams=NUM_BEAMS,
    length_penalty=LENGTH_PENALTY,
)

for i in range(min(5, len(train_sample_preds))):
    print(f"\n{'='*70}")
    print(f"ID:    {train_sample_ids[i]}")
    print(f"Text:  {train_sample_texts[i][:100]}")
    print(f"Gold:  {train_sample_golds[i][:120]}")
    print(f"Pred:  {train_sample_preds[i]}")

---

## Summary — v2 vs v1

| Feature | v1 | v2 |
|---|---|---|
| **Model** | T5-small (60M) | **T5-base (220M)** |
| **Batch size** | 8 | 8 × 4 = **32 effective** |
| **Learning rate** | 3e-4 | **2e-4** |
| **LR scheduler** | linear | **cosine annealing** |
| **Warmup** | 200 steps | **10% ratio** |
| **Epochs** | 10 | **15** |
| **Beam search** | 4 beams | **5 beams + length penalty** |
| **Text preprocessing** | None | **Whitespace normalization** |
| **Saved Model** | `aste_t5_model/best/` | `aste_t5base_model/best/` |

### When test data arrives:
1. Upload `test.jsonl` to your Kaggle dataset
2. Run **Section 7**
3. Download `predictions.jsonl` from `/kaggle/working/`